# Topic: Window Function Ranking

## Definition (30-second explanation)
* Window functions compute values across a set of rows related to the current row, without collapsing them into a single output row like `GROUP BY` does.
* Ranking window functions—`ROW_NUMBER()`, `RANK()`, and `DENSE_RANK()`—assign positional numbers to rows within a defined partition based on a specific order.

## Why Interviewers Ask This
* To test your ability to write sophisticated analytical queries (like Top-N reporting) without relying on inefficient self-joins.
* To verify you know how to deduplicate messy datasets safely.
* To see if you understand the precise logical execution order of SQL (e.g., filtering window functions requires a CTE).

## Core Concepts
* **OVER():** Defines the window. Read it as "for each group, in this order."
* **PARTITION BY:** Divides the result set into groups. Without it, the function computes across the entire table.
* **ORDER BY:** Defines the logical order of evaluation within each partition.
* **Execution Order:** Window functions are evaluated *after* `WHERE` and `GROUP BY` clauses.

## When to Use
* **ROW_NUMBER():** Best for deduplication, pagination, or when you need a strict, arbitrary tie-breaker.
* **RANK():** Best for leaderboards where gaps in ranking are mathematically required after ties.
* **DENSE_RANK():** Best for "Top-N" queries (e.g., Top 3 salaries) where all tied individuals should be included without skipping the next rank.
* **NTILE(n):** Best for percentile bucketing (e.g., dividing populations into quartiles/deciles).

## Advantages
* Keeps the row-level detail intact while providing aggregate/ranked insights.
* Significantly more readable and computationally efficient than correlated subqueries.

## Limitations
* Cannot be used directly inside a `WHERE` clause; requires wrapping in a Common Table Expression (CTE) or subquery to filter by the rank.

## Common Comparisons
| Function | Handles Ties By | Skips Next Number? | Best Use Case |
| :--- | :--- | :--- | :--- |
| **ROW_NUMBER()** | Giving unique incremental numbers | No | Deduplication, Pagination |
| **RANK()** | Giving the exact same rank | Yes (e.g., 1, 2, 2, 4) | Leaderboards |
| **DENSE_RANK()** | Giving the exact same rank | No (e.g., 1, 2, 2, 3) | Top-N reporting |

## Common Interview Traps
* **Forgetting PARTITION BY:** Causes the ranking to apply globally across the whole table instead of within the intended groups.
* **Using ROW_NUMBER for Top-N:** If two users tie for 3rd place, `ROW_NUMBER` arbitrarily cuts one off. Always use `DENSE_RANK` for Top-N unless instructed otherwise.
* **Filtering in the WHERE clause:** Writing `WHERE ROW_NUMBER() = 1` directly will throw a syntax error.

## Python / SQL Syntax
```sql
-- Standard Syntax Pattern
SELECT 
    department,
    salary,
    ROW_NUMBER() OVER(PARTITION BY department ORDER BY salary DESC) as rn,
    RANK() OVER(PARTITION BY department ORDER BY salary DESC) as rnk,
    DENSE_RANK() OVER(PARTITION BY department ORDER BY salary DESC) as dense_rnk
FROM employees;
```

## 45-Second Interview Answer
"Window function rankings assign sequential numbers to rows within specific partitions without collapsing the dataset. The three main functions handle ties differently: ROW_NUMBER always gives a unique integer, which is perfect for deduplication. RANK gives tied rows the same number but skips the subsequent numbers, useful for competitive leaderboards. DENSE_RANK gives tied rows the same number without skipping, making it the standard choice for Top-N analytical queries. Because window functions evaluate after the WHERE clause, I always wrap them in a CTE when I need to filter on the resulting rank."